# 1. Units and conventions

Before defining a model, put observations and physical parameters in the
units expected by JeansPy. Arrays contain plain numerical values: the model
does not infer a unit from an array or convert an Astropy quantity for you.


This notebook is self-contained. Install JeansPy with the `numpyro_cpu` and
`plotting` extras as described in the installation guide, then select that
environment as your Jupyter kernel and run cells from top to bottom.
Saved outputs are an example run; timings and short-chain results can vary.

## Prepare observations

For spherical inference, each row represents a star. `R_pc` is the projected
distance from the galaxy center, not its three-dimensional radius. Supply
the measured LOS velocity and its standard error in the same velocity frame.

In [1]:
import numpy as np

data = dict(
    R_pc=np.array([30., 100., 300.]),
    vlos_kms=np.array([-4., 2., 8.]),
    e_vlos_kms=np.array([2., 2., 1.5]),
)
assert all(value.shape == (3,) for value in data.values())
assert np.all(data["R_pc"] > 0) and np.all(data["e_vlos_kms"] >= 0)
import pandas as pd
pd.DataFrame(data)

,R_pc,vlos_kms,e_vlos_kms
0,30.0,-4.0,2.0
1,100.0,2.0,2.0
2,300.0,8.0,1.5


The arrays must be finite, nonempty, one-dimensional and have identical lengths;
errors must be nonnegative. Convert angular separations to projected pc using
the adopted galaxy distance before building this dictionary. That distance
and the coordinate origin are inputs to your analysis.

## Read parameter names and returned quantities

| Quantity | Meaning and units |
| --- | --- |
| `R_pc`, `r_pc` | Projected and intrinsic radius, respectively, in pc |
| `re_pc`, `rs_pc`, `r_t_pc` | Tracer scale, halo scale and halo cutoff in pc |
| `rhos_Msunpc3` | Halo mass density scale in solar masses / pc³ |
| `vmem_kms`, `vlos_kms`, `e_vlos_kms` | Systemic velocity, observation and standard error in km/s |
| `sigmalos2` | Intrinsic LOS variance in (km/s)² |
| `density_2d`, `density_3d` on a normalized tracer | Number density in pc⁻² and pc⁻³ |
| `mass_density_3d`, `enclosed_mass` on a halo | Mass density in solar masses / pc³ and mass in solar masses |

For the Plummer tracer, `re_pc` is the projected half-light radius. Other
profiles can use a different scale convention: in particular `Exp3dModel`
uses an exponential scale length. Check the selected class's API entry.

A unit-normalized tracer supplies the spatial weighting of the stars. It
does **not** add stellar mass to the gravitational potential. The halo provides
the enclosed mass used by the spherical solver.

## Respect geometry and array shapes

Spherical LOS predictions require finite **positive** radii; the central limit
at `R_pc=0` is not implemented. NumPy returns a scalar for a scalar radius and
a one-dimensional array for an array. JAX always returns a one-dimensional
array, including a length-one array for a scalar radius.

Axisymmetric models instead take signed sky coordinates `x_pc`, `y_pc`, allow
the projected center and broadcast the coordinate arrays. Inclination is in
radians. Spherical $\beta=1-\sigma_\theta^2/\sigma_r^2$ and cylindrical
$\beta_z=1-\sigma_z^2/\sigma_R^2$ describe different tensors; substituting one
for the other changes the model.

## Separate physical validity from numerical success

An invalid NumPy input can raise an exception; invalid dynamic JAX values can
produce NaN and be rejected by the likelihood. A finite prediction alone does
not establish that a nonnegative phase-space distribution exists. Likewise,
doubling quadrature resolution checks numerical stability, while sampler
diagnostics assess sampling. Neither alone validates the physical model.

The [model contract](../guides/contracts.md) collects the full shape, geometry
and error conventions. Next: [choose a backend](backends.ipynb).